In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-10: Preference pairs and DPO

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-9's filters give rejected completions weight 0. Can a rejected completion teach? A pair uses both. This session: DPO.**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples (RLLM-7), 2 who supplies the weights (a reference or a reward), 3 how fine (RLLM-8). Here: one new method
- **Position**: session 10 of 14, after RLLM-9 (Sample, filter, imitate: self-training)
- **Goal of Part 2**: a **map** that places any method by the three axes

<img src="images/series_map_10.png" width=55%>


## This session: RLLM-10

- **Question**: can the **rejected** completion teach, and what does the **DPO** loss do with a pair?
- **By the end you can**:
  - say what a **preference pair** is and what DPO's **pair weight** (one number per pair) does: up on the winner's tokens, down on the loser's
  - write the **margin** (how far the model moved toward the winner, relative to the frozen **reference model**) and the loss, and compute them
  - say in what sense DPO equals RLHF: the **solutions** agree under stated assumptions, not the updates
  - place DPO with the **three axes** (a **reward**, a pair), saying honestly whose samples it uses
  - say what DPO's signal cannot see: only an **ordering**, at **stored states**, as a **difference** not a level


## Map: the pipeline of this session, one section per box

> **DPO is a short pipeline: a pair, a frozen reference model and a margin, a loss, and the question of where the RL went. Four more sections place it and test it.**

<img src="images/rllm10_map.png" alt="the DPO pipeline pair, reference model and margin, loss, origin; then the sections comparison, placement, variant and limit" width=54%>

- This is the **plan**; each box names the section that covers it
- **Pair**, **Reference model**, **Loss**, **Origin**: the pipeline, in the order it runs
- **Comparison**, **Placement**, **Variant**, **Limit**: how DPO relates to RLLM-9, to the axes, to its online form, and where it stops
- *Next: the pair, the signal DPO uses*


## Pair: a pair is a different signal, a contrast within one prompt

<!-- seed-pay: weight-zero-rejected -->
> **A preference pair is a prompt with a winner and a loser. DPO raises the winner and lowers the loser in one update, with a weight that depends on the model's current margin.**

- **From RLLM-9**: a filter gives a rejected completion **weight 0**: no *explicit* push down on it
  - a pair is a different signal, a **contrast within one prompt**, not a fix for that
- **Axis 2** (our reading): DPO sits on the **reward** side
  - the frozen reference model is an **anchor** (a KL anchor, Reference model section), not the weight supplier: the pair supplies the weights
  - a reward is a judgment of a whole completion: a score or a preference pair
  - careful: this **reference model** (a frozen copy of the start) is not RLLM-8's axis-2 "reference" column, which supplies per-token weights
- **Preference pair** $(\x,\y^+,\y^-)$: a prompt, a **winner** (the literature says *chosen*) and a **loser** (*rejected*)
  - it says which is better, **not by how much** (RLLM-2)
- Pairs come from people, a judge model, or a reward that ranks sampled completions: best against worst
- **DPO** (direct preference optimization): one **supervised loss** on pairs, with no reward model and no RL loop
- *Next: what DPO does with a pair, in one sentence*


## Pair: up on the winner, down on the loser, scaled by how wrong the margin is

> **DPO puts $+c$ on every token of the winner and $-c$ on every token of the loser. Let $z$ be how well the model already separates this pair; the pair weight $c=\beta\,\sigma(-z)$ is large when it does not.**

<img src="images/dpo_pair_weight.png" alt="left: the pair weight against the margin; right: token weights on one pair at three margins" width=65%>

- **Reference model**: a frozen copy of the model as it was before training (next section)
- **Margin** $z$: how far the model has moved toward the winner, compared with that frozen copy
  - $z=0$ at the start
- $\beta$ is RLLM-2's KL coefficient; $\sigma$ is the logistic sigmoid
- $c$ is the weight the loss's gradient puts on each token (skipped aside "the pair weight $\beta\,\sigma(-z)$ on a pair")
- Same form as every RLLM-8 update: a weight on each token
  - unlike RLLM-9's, $c$ depends on the **model's current state**
- *Next: the reference model and the margin*


## Reference model: a frozen copy, and the margin

*Why this section: a winner-up, loser-down push needs a zero point and a strength. The frozen reference model supplies both: the margin is measured against it, and $\beta$ says how tightly to stay near it.*

- **Reference model** $\pi_{\text{ref}}$: a frozen copy of the starting model, never updated (the KL anchor of RLLM-2 and RLLM-4)
- **Log-ratio** of a completion: $\log\dfrac{\pi_\theta(\y\mid\x)}{\pi_{\text{ref}}(\y\mid\x)}$, with log-probabilities of the whole completion (summed over tokens)
- **Implicit reward**: $\hat{\rew}_\theta(\x,\y)=\beta\times$ log-ratio
  - implicit: it is a function of the parameters $\theta$, not a separate network
  - a reward **only up to any term that depends on the prompt alone**
  - such a term cancels in a pair (more: Origin)
  - **Three senses of "reward":** the pair supplying the weights (axis 2), this implicit reward (just $\beta\times$ log-ratio), Bradley-Terry strength (RLLM-2)
  - no separate reward **model** runs at training time
- **Margin** of a pair: $z=\hat{\rew}_\theta(\x,\y^+)-\hat{\rew}_\theta(\x,\y^-)$
  - $z>0$: the model has moved toward the winner, relative to the reference model
  - $\beta$ multiplies the log-ratio inside $z$: large $\beta$ means a small move from the reference separates winner from loser, so the model stays closer
  - the KL link: skipped aside "DPO's loss from the KL-penalized RL objective"
- The reference model **sets how large the pair weight is** (through $z$), not its direction
  - the log-ratios **sum over the whole completion**, so a long completion has more terms and completion length matters
  - on **separable** pairs (every winner outranks its loser) DPO can keep pushing and overfit (our reading, RLLM-2)
- The reference model is frozen, so its log-probabilities can be **computed once and cached**
- *Next: the loss that uses the margin*


## Loss: a classifier on the margin

*Why this section: the margin acts as a reward gap, and RLLM-2 showed how to fit rewards to pairs. We reuse that loss with the margin as the gap.*

- **Loss** for one pair: $\loss=-\log\sigma(z)$. It falls as $z$ grows, and is $\log 2\approx0.693$ at $z=0$
- It is RLLM-2's Bradley-Terry loss with the margin in place of the reward gap
  - $\sigma(z)$ is the probability the model gives to "the winner is preferred"
- Unusual: the classifier's parameters **are** the language model's parameters
- Its gradient is the punchline's push: $+c$ on the winner, $-c$ on the loser (skipped aside "the pair weight $\beta\,\sigma(-z)$ on a pair")

```python
# F = torch.nn.functional
def dpo_loss(logp_w, logp_l, ref_logp_w, ref_logp_l, beta):
    # each argument: log-probability of a whole completion (summed over its tokens)
    ratio_w = logp_w - ref_logp_w          # log-ratio of the winner
    ratio_l = logp_l - ref_logp_l          # log-ratio of the loser
    z = beta * (ratio_w - ratio_l)         # the margin
    return -F.logsigmoid(z).mean()
```

- *Next: the loss on one worked pair*


## Loss: a worked pair (synthetic numbers, $\beta=0.1$)

Reference-model log-probabilities: $-40$ for $\y^+$ and $-38$ for $\y^-$ (it prefers the loser).

| model | log-prob y+ | log-prob y- | log-ratio y+ | log-ratio y- | margin z | loss | pair weight |
|---|---|---|---|---|---|---|---|
| start: the model equals the reference model | -40 | -38 | +0 | +0 | +0.0 | 0.693 | 0.0500 |
| after training: y+ up, y- down | -38 | -41 | +2 | -3 | +0.5 | 0.474 | 0.0378 |
| far right: margin z = +2 | -30 | -48 | +10 | -10 | +2.0 | 0.127 | 0.0119 |
| far wrong: margin z = -2 | -60 | -38 | -20 | +0 | -2.0 | 2.127 | 0.0881 |

- **Start:** $z=0$, loss $\log2$, pair weight $c=\beta/2$
- **After training:** the margin grew, loss and weight fell. Only the margin is guaranteed to grow (one pair, first order)
- **Far wrong ($z=-2$):** weight 0.088, against 0.012 at $z=+2$: a pair the model gets wrong gets most of the push
- *Next: where the RL went*


## Origin: where the RL is

*Why this section: DPO has a reward and a KL coefficient but runs no RL. What is left of the RL?*

> **DPO runs no RL; the RL is in the loss's origin. RLLM-2's KL objective has a best policy that makes the reward a log-ratio.**

<img src="images/dpo_skips_stages.png" alt="RLHF fits a reward model then runs RL; DPO skips both stages" width=70%>

- **Not RL training**: a supervised loss on fixed pairs, with no sampling, no reward model, no baseline
- RLHF fits a reward model, then runs RL. DPO **skips both stages**. Next: why that is allowed

## Origin: the bridge from RLHF to DPO

- RLLM-2's objective: $\max{\pi}\;\Exp{\y\sim\pi}[\,\rew(\x,\y)\,]-\beta\,\KL(\pi\,\Vert\,\pi_{\text{ref}})$
- Its best policy (RLLM-2): $\pi^*(\y\mid\x)\;\propto\;\pi_{\text{ref}}(\y\mid\x)\,e^{\rew(\x,\y)/\beta}$
- **Solve for the reward**: $\rew(\x,\y)=\beta\log\dfrac{\pi^*(\y\mid\x)}{\pi_{\text{ref}}(\y\mid\x)}+{}$ a term depending **only on the prompt**
  - that term is $\beta\log Z(\x)$, the **normaliser** of $\pi^*$: it does not depend on the completion
  - this is the implicit reward of the Reference model section, with $\pi_\theta$ in place of $\pi^*$: a reward **only up to the prompt term**
- In a pair's Bradley-Terry difference the prompt-only term **cancels** (both completions share the prompt), so no reward model is needed
  - the steps are in the skipped aside "DPO's loss from the KL-penalized RL objective"
  - it rests on pairs following Bradley-Terry and on a model that can represent $\pi^*$ (the model class can express the optimal policy exactly)
> **The prompt-only term cancels, so the loss never needs the reward or the normaliser.**

- The equivalence is between **solutions** (where the loss is minimized), not between updates: next slide
- *Next: how this equivalence differs from RLLM-9's*


## Comparison: updates versus solutions, RLLM-9 and DPO side by side

> **Both turn an RL-style objective into a supervised computation. RLLM-9's equivalence is of one expected update; DPO's is of where the loss is minimized.**

| | **RLLM-9**: filter, then SFT | **DPO** |
|---|---|---|
| RL-style objective | probability of clearing the bar | reward minus $\beta\times$KL to the reference model |
| What you run | SFT on the kept samples | a classification loss on pairs |
| Equivalence between | **updates**: in expectation, one step | **solutions**: where the loss is minimized (if pairs follow Bradley-Terry and the optimum is representable) |
| Anchored to the start? | **no** KL term | **yes**: KL term, coefficient $\beta$ |
| Needs the current model's samples? | **yes** | **no**: stored pairs (the online variant, in the Variant section, does) |

- Neither is simply "RL" or "just SFT": ask **which objective it equals, and in what sense**
- *Next: where DPO sits on the three axes*


## Placement: the three axes, said honestly

1. **Whose samples?** Usually **someone else's, earlier**: stored pairs, **off-policy**
   - the online variant, resampled each round, is **near on-policy** (RLLM-7's dial)
2. **Who supplies the weights?** A **reward**: a comparison of the pair (our reading)
   - a pair is a judgment of whole completions, which is what a reward gives
   - the weights then fall on tokens: $+c$ on the winner's, $-c$ on the loser's, a push-down (RLLM-8)
3. **How fine?** **One per completion** (set by the pair): the same weight $\pm c$ on every token of each

- *Next: the map, with DPO added*


## Placement: DPO on the map

> **DPO joins the map in the others'-samples row, on the reward side. Everything placed in RLLM-8 and RLLM-9 stays.**

<img src="images/third_map.png" alt="the map with DPO filled and highlighted in the others'-samples row, reward column; the hybrid box from RLLM-9 in normal color; the OPD slot still dashed" width=46%>

- New in this session: **DPO** only; the hybrid box is RLLM-9's
- Not in the reference column: a comparison, not a token, supplies the weights
- The online variant (pairs resampled each round) is nearer the own-samples row. The map shows stored pairs
- Placement is **our reading** of how DPO is built, not a statement of whether it works
- *Next: a variant that resamples its pairs*


## Variant: online DPO samples and ranks its own pairs

> **The main story used stored pairs, made once. The online variant makes pairs from the model's own samples, round after round.**

<img src="images/sample_rank_pair_loop.png" alt="sample, reward, pair best vs. worst, contrast relative to the frozen reference model; optionally repeat" width=70%>

- Steps 1 and 2 are RLLM-9's loop: **sample** $G$ completions, compute each one's **reward**
- Step 3 **pairs** best against worst, where RLLM-9's filter kept only the best. Step 4 is the DPO loss
- Repeating on the newly trained model moves DPO toward **on-policy** (RLLM-7's dial, Guo et al., 2024)
- A ranking by a proxy reward can be gamed (RLLM-2)
- A prompt whose $G$ completions tie gives **no pair**
- *Next: where DPO's signal stops*


## Limit: an ordering, at stored states, as a difference

> **DPO sees only a margin between two whole completions, at fixed states.**

<img src="images/dpo_difference_not_level.png" alt="log-probabilities of the winner and the loser under the reference model and two trained models" width=50%>

- **Ordering only**: how much better the winner is, or whether both are bad, is invisible
- **Fixed states** (for stored pairs): only the stored completions' prefixes. Behavior no pair covers is not pushed on
- **A difference, not a level**: in trained model B both log-probabilities fell, yet its margin is larger than A's (Pal et al., 2024)
- Even the margin grows only for **one pair, a small step, first order** (skipped aside "the pair weight $eta\,\sigma(-z)$ on a pair")
- *Next: questions to predict before you answer*


## Think about

*State a prediction before you work each one.*

1. Online DPO ranks $G=8$ samples per prompt, and one prompt's completions **all earn the same reward**
   - what happens to that prompt? Which of RLLM-9's two methods uses it?
2. **Method Z**: sample 8 completions, a judge ranks them, DPO on the (best, worst) pair
   - place it with the three axes
   - what changes if a filter keeps the **top 2** for SFT (RLLM-9)?
3. Pairs made **once, up front** versus **resampled each round**
   - where does each sit on RLLM-7's staleness dial; what would you measure?
4. The loss sees only a **margin**
   - design a small check of whether the winner's own log-probability **rises** during training
   - what would you log?


## What we established, and what it sets up

- **Established**: a **pair weight** raises the winner's tokens and lowers the loser's: a rejected completion teaches, as a margin. Solutions match RLHF's (Bradley-Terry assumed)
- **Limit of this session**: an **ordering** of whole completions, at **stored states**, as a **difference**: the winner's own probability can fall
- **Sets up**: RLLM-11, On-policy distillation: the mechanism
  - **DPO only sees a margin at stored states. Can we get a dense signal (a number per token) at the model's OWN states?**


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*](https://arxiv.org/abs/2305.18290): abstract, introduction, section 4 loss
  - [Rasul, Belkada and von Werra, *Fine-tune Llama 2 with DPO*, Hugging Face blog, 2023](https://huggingface.co/blog/dpo-trl): a short worked example of DPO in practice
- **Additional** (5 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


Where the names come from, if you want to read further:

- **DPO**: Rafailov et al. (2023), *Direct Preference Optimization: Your
  Language Model Is Secretly a Reward Model* (NeurIPS 2023)
- **Bradley-Terry**: Bradley and Terry (1952), a model of paired
  comparisons
- **Reward model from pairs, then RL**: Christiano et al. (2017),
  *Deep Reinforcement Learning from Human Preferences*
- **Preferred likelihood can fall**: Pal et al. (2024), *Smaug: Fixing
  Failure Modes of Preference Optimisation with DPO-Positive*
- **Resampling pairs from the current model**: Guo et al. (2024),
  *Direct Language Model Alignment from Online AI Feedback*


### Aside (1 of 3), justification: where "sigmoid of a difference" comes from

*For your own review; we will not work through it in session.*

**Setup.** Items $i, j$ have **strengths** $s_i, s_j$. We model $P(i \succ j)$, the probability that $i$ is preferred (Bradley and Terry, 1952).

1. **Assume** preference probability is proportional to exponentiated
   strength: $P(i \succ j)=\dfrac{e^{s_i}}{e^{s_i}+e^{s_j}}$.
   *(This is the model's one assumption.)*
2. **Divide** by the opposite probability:
   $\dfrac{P(i\succ j)}{P(j\succ i)}=e^{s_i-s_j}$.
   *(The common denominator cancels.)*
3. So the **log-odds** is $s_i-s_j$, the difference of the strengths.
   *(Take the log of step 2.)*
4. **Invert** the log-odds to a probability:
   $P(i\succ j)=\sigma(s_i-s_j)$, with $\sigma(u)=1/(1+e^{-u})$.
   *(Definition of the sigmoid; it is not an extra assumption.)*

**Result.** "Sigmoid of a difference" *is* Bradley-Terry; reward models fit to human pairs use it (Christiano et al., 2017).


### Aside (2 of 3), justification: DPO's loss from the KL-penalized RL objective

*For your own review; we will not work through it in session. The slide "The bridge from RLHF to DPO" gives the idea.*

**Setup.** $\rew(\x,\y)$ is a reward, $\pi_{\text{ref}}$ the frozen
reference model, $\beta>0$ the KL coefficient.

1. The **RL problem**: maximize
   $\Exp{\y\sim\pi}[\,\rew(\x,\y)\,]-\beta\,\KL(\pi\,\Vert\,\pi_{\text{ref}})$
   over policies $\pi$. *(The standard KL-penalized objective.)*
2. Its **optimal policy** is
   $\pi^*(\y\mid \x)=\dfrac{\pi_{\text{ref}}(\y\mid \x)\,e^{\rew(\x,\y)/\beta}}{Z(\x)}$,
   with $Z(\x)$ a normalizer that does not depend on $\y$.
   *(A known closed form for this objective.)*
3. **Solve for the reward**:
   $\rew(\x,\y)=\beta\log\dfrac{\pi^*(\y\mid \x)}{\pi_{\text{ref}}(\y\mid \x)}+\beta\log Z(\x)$.
   *(Take the log of step 2 and rearrange.)*
4. Put this into the **Bradley-Terry difference** $\rew(\x,\y^+)-\rew(\x,\y^-)$ from
   aside 1.
5. The two $\beta\log Z(\x)$ terms are the **same** (same prompt) and **cancel**.
   *(The unknown normalizer is never computed.)*
6. What remains is $z$, with $\pi_\theta$ standing in for $\pi^*$.
   Maximizing the **likelihood of the observed preferences**, $\sigma(z)$,
   is minimizing $-\log\sigma(z)$: the DPO loss.
   *(Maximum likelihood on observed winners.)*

**What this does and does not say.**

- It shows the loss is the **right classifier if** the pairs follow
  Bradley-Terry with *some* reward and the model class can represent
  the optimum
- It says **nothing** about how well plain gradient descent gets there,
  or how the loss behaves when those assumptions fail


### Aside (3 of 3), justification: the pair weight $\beta\,\sigma(-z)$ on a pair

*For your own review; we will not work through it in session.*

1. $\loss=-\log\sigma(z)$. *(The loss.)*
2. $\dfrac{d}{dz}\big[-\log\sigma(z)\big]=-\sigma(-z)$.
   *(Derivative of $-\log\sigma$, using $1-\sigma(z)=\sigma(-z)$.)*
3. $z$ depends on $\theta$ only through
   $\beta\,[\log\pi_\theta(\y^+\mid \x)-\log\pi_\theta(\y^-\mid \x)]$.
   *(The reference-model terms are constants for the parameters.)*
4. So
   $\nabla_\theta\loss=-\beta\,\sigma(-z)\,\big[\nabla_\theta\log\pi_\theta(\y^+\mid \x)-\nabla_\theta\log\pi_\theta(\y^-\mid \x)\big]$.
   *(Chain rule.)*

5. Write $\g^+=\nabla_\theta\log\pi_\theta(\y^+\mid \x)$ and
   $\g^-=\nabla_\theta\log\pi_\theta(\y^-\mid \x)$. A small step on **one pair** is
   $\Delta\theta=\eta\,\beta\,\sigma(-z)\,(\g^+-\g^-)$.
   *(Gradient descent, step size $\eta$.)*
6. To first order, the **margin** $z$ changes by
   $\Delta z=\beta\,(\g^+-\g^-)\cdot\Delta\theta=\eta\,\beta^2\,\sigma(-z)\,\Vert \g^+-\g^-\Vert^2\ge0$.
   *(A squared norm.)*
7. But the **winner alone** changes by
   $\g^+\cdot\Delta\theta\propto\Vert \g^+\Vert^2-\g^+\cdot \g^-$,
   and the **loser alone** by $\g^+\cdot \g^--\Vert \g^-\Vert^2$.
   *(Dot products.)*
8. Either can have the "wrong" sign when $\g^+$ and $\g^-$ are strongly
   aligned and of unequal size, for instance when the two completions
   share most of their tokens.

**Result, in words.**

- The pair gets **one shared pair weight** $\beta\,\sigma(-z)$, large when
  $z<0$ and small when $z>0$
- What holds is **one pair, a small step, first order**: the *margin* grows;
  a batch of pairs or an adaptive optimizer is not covered
- Each log-probability separately can move either way: "a difference, not a level"
